In [1]:
from langchain_community.document_loaders import PyPDFLoader
import os
print(os.getcwd()) # 현재 위치 출력

c:\Users\Admin\hipython\llm


In [2]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [6]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

In [7]:
# 기존 인덱스 없으면 생성
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"{INDEX_NAME} 생성 완료")
else:
    print(f"{INDEX_NAME} 이미 존재함")

# 상태 확인
f_index = pc.Index(INDEX_NAME)
print(f_index .describe_index_stats())

finance-bok 생성 완료


c:\Users\Admin\miniconda3\envs\langchain_rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [11]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "./data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")
print(f"\n첫 페이지 내용 (앞 300자):\n{docs[0].page_content[:300]}")
print(f"\n메타데이터: {docs[0].metadata}")

총 페이지 수: 93

첫 페이지 내용 (앞 300자):
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

In [13]:
splits = splitter.split_documents(docs) #원본 문서의 메타 정보를 가져와 설정

In [14]:
print(f"총 청크 수: {len(splits)}")
print(f"\n첫 번째 청크:\n{splits[0].page_content}")
print(f"\n메타데이터: {splits[0].metadata}")

총 청크 수: 249

첫 번째 청크:
경제전망 
 Indigo Book 
2026년 2월 
 
 
 
 
성장 2%대 반등, 부문별 온도차

메타데이터: {'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2026-02-26T16:48:20+09:00', 'author': 'A11', 'moddate': '2026-03-31T15:12:50+09:00', 'source': './data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf', 'total_pages': 93, 'page': 0, 'page_label': '1'}


In [15]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [16]:
# 업서트
INDEX_NAME = "finance-bok"
NAMESPACE  = "bok-ns1"

print("업서트 시작...")
vectorstore = PineconeVectorStore.from_documents(
    documents=splits,
    embedding=embedding_model ,
    index_name=INDEX_NAME,
    namespace=NAMESPACE
)
print(f"업서트 완료 — 총 {len(splits)}개 청크")

업서트 시작...
업서트 완료 — 총 249개 청크


In [17]:
from pinecone import Pinecone

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])
index = pc.Index(INDEX_NAME)
stats = index.describe_index_stats()
print(stats)

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-ns1': {'vector_count': 249}},
 'total_vector_count': 249,
 'vector_type': 'dense'}


# retriever 생성, 호출

In [19]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore.from_existing_index(
    index_name=INDEX_NAME,
    embedding=embedding_model,
    namespace=NAMESPACE
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3, "namespace": NAMESPACE}
)

In [22]:
# 검색 테스트
queries = [
    "2026년 경제성장률 전망은?",
    "변동성 전망은?",
    "리스크 요인은?"
]

In [23]:
for query in queries:
    print(f"[ 질문 ] {query}")
    results = retriever.invoke(query)
    for i, r in enumerate(results):
        print(f"  {i+1}. (p.{r.metadata.get('page', '')}) {r.page_content[:100]}")
    print()

[ 질문 ] 2026년 경제성장률 전망은?
  1. (p.43.0) 30 
 
<경제성장 전망1)> 
(전년동기대비, %) 
  2024 2025 2026e) 2027e) 
연간 상반 하반 연간 상반 하반 연간 연간 
GDP 성장률 2.0 0.3 
  2. (p.35.0) 22 
 
2. 거시경제 전망 
 
 
 
 
경제성장 
 
2-1. 금년중 국내경제는 美관세 영향, 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한
  3. (p.6.0) < 요약 1/8 > 
 
  
경제전망 요약 
 올해 우리 경제는 美관세 영향과 건설투자의 더딘 회복에도 불구하고 반도체 경기 
개선세 확대, 예상보다 양호한 세계경제 흐름 등에

[ 질문 ] 변동성 전망은?
  1. (p.89.0) 전망오차(①)로부터 설명되지 않는지, 전망치의 변화가 실제치의 변화를 잘 나타내는지
(②)를 평가
  2. (p.84.0) ➌ 효율성efficiency : 전망시 가용정보가 충분히 반영되는지? 
 
15. 연간 전망에 대한 효율성 검정 결과, 성장 전망치는 효율성 조건을 일부 충족하지 
못했으나 물가 
  3. (p.76.0) 63 
 
장률 전망의 오차s-RMSE 기준가 0.68로 국내외 주요 예측기관(0.69~0.95)과 비교해 대
체로 비슷하거나 작은 수준이었다.[그림7] 최근에는22년 이후 분기 

[ 질문 ] 리스크 요인은?
  1. (p.52.0) 국내외 여건 및 전망 
3. 전망의 리스크 평가 
 
 
 
 
 
 
 
 
 
 
목 차  
주요 리스크 요인 ....................................
  2. (p.21.0) 해 관세정책의 불확실성이 커진 가운데 AI 투자 가속화, 주요국의 대미투자 증가 
등이 상방리스크로, 급격한 AI 투자 조정, 무역분쟁 심화 등이 하방리스크로 각각 
작용할 것으로
  3. (p.53.0) 40 
 
3. 전망의 리스크 평가 
    
주요 리스크 요인 
 
3-1. 향후 성장 전망경로에

# RAG CHAIN

In [24]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_openai import ChatOpenAI

llm    = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

In [25]:
rag_chain = (
    RunnableParallel(
        context=retriever,
        question=RunnablePassthrough()
    )
    | prompt
    | llm
    | parser
)

In [26]:
question = '수출 전망은 어떻게 되나요?'
rag_chain.invoke(question)

'수출은 견조한 흐름을 이어갈 것으로 전망됩니다. 2024년 수출은 6,836억 달러에서 시작하여 2025년에는 7,093억 달러, 2026년에는 7,952억 달러로 증가할 것으로 예상됩니다.'

In [27]:
question_1 = '변동성 전망은 어떻게 되나요?'
rag_chain.invoke(question_1)

'보고서에서 확인되지 않습니다.'

In [29]:
question_2 = '리스크는 어떻게 되나요?'
print(rag_chain.invoke(question_2))

리스크는 상방리스크와 하방리스크로 나눌 수 있습니다.

**상방리스크:**
- 반도체 경기 상승세 추가 확대
- 국내외 정부의 경기부양책 강화
- 통상환경 불확실성 완화
- 미국 빅테크 투자 견조한 흐름 전망
- 주요국 재정지출 확대 기조

**하방리스크:**
- 미국 관세정책 불확실성 추가 증대
- 국제금융시장 불안
- AI 과잉투자 및 주요국 재정 우려
- 비IT부문 부진 심화
- 지정학적 불안 심화
- 글로벌 원유 공급과잉 심화
- 고환율 지속
- 정부의 물가안정대책 강화


In [30]:
questions = [
    "주요 대외 불확실성 지표에 대해 설명해주세요.",
    "원/달러 환율 및 시장금리에 대해 설명해주세요.",
    "AI과잉 투자에 대한 상황과 우려에 대해 자세하게 설명해주세요.",
    "2026년 2월 경제전망 요약표를 설명해주세요.",
    "거시경제 전망을 요약해주세요.",
    "전망의 리스크 평가를 자세하게 설명해주세요."
]

for q in questions:
    print(f"[ Q ] {q}")
    answer = rag_chain.invoke(q)
    print(f"[ A ] {answer}")
    print()

[ Q ] 주요 대외 불확실성 지표에 대해 설명해주세요.
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 원/달러 환율 및 시장금리에 대해 설명해주세요.
[ A ] 원/달러 환율은 거주자 해외증권투자와 엔화 동조 등으로 변동성이 높아진 가운데 최근 1,400원대 중반 수준으로 낮아졌습니다. 시장금리는 기준금리 인하 기대 약화와 수급 요인 등으로 상승세를 나타내고 있습니다.

[ Q ] AI과잉 투자에 대한 상황과 우려에 대해 자세하게 설명해주세요.
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 2026년 2월 경제전망 요약표를 설명해주세요.
[ A ] 2026년 2월 경제전망 요약표는 다음과 같습니다:

- **GDP 성장률**: 2024년 2.0%, 2025년 1.0%, 2026년 2.0% (+0.2), 2027년 1.8% (-0.1)
- **소비자물가 상승률**: 2024년 2.3%, 2025년 2.1%, 2026년 2.2% (+0.1), 2027년 2.0%
- **근원물가**: 2024년 2.2%, 2025년 1.9%, 2026년 2.1% (+0.1), 2027년 2.0%

전망은 반도체 경기 개선과 양호한 세계경제 흐름에 힘입어 이루어졌으며, 소비자물가는 일부 품목의 비용 상승 압력으로 인해 당초 예상보다 소폭 상승할 것으로 보입니다.

[ Q ] 거시경제 전망을 요약해주세요.
[ A ] 보고서에서 확인되지 않습니다.

[ Q ] 전망의 리스크 평가를 자세하게 설명해주세요.
[ A ] 보고서에서 확인되지 않습니다.



In [33]:
# RAG 답변 vs 일반 LLM 답변 비교
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm_base = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]

for q in test_questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
[ 일반 LLM ] 2026년 GDP 성장률 전망치는 여러 경제 기관과 연구소에 따라 다를 수 있으며, 특정한 수치를 제공하기 위해서는 최신 경제 보고서나 예측 자료를 참조해야 합니다. 일반적으로 이러한 전망치는 경제 상황, 정책 변화, 글로벌 경제 동향 등에 따라 변동할 수 있습니다. 
[ RAG 답변 ] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 2026년 소비자물가 상승률 전망치는?
[ 일반 LLM ] 2026년 소비자물가 상승률에 대한 구체적인 전망치는 여러 경제 기관이나 정부의 경제 전망에 따라 달라질 수 있습니다. 일반적으로 이러한 전망치는 경제 성장, 통화 정책, 국제 유가, 공급망 문제 등 다양한 요인에 의해 영향을 받습니다. 

정확한 수치를 알고 싶다면,
[ RAG 답변 ] 2026년 소비자물가 상승률 전망치는 2.0%입니다.

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
[ 일반 LLM ] 2026년의 수출 전망 금액에 대한 구체적인 수치는 여러 요인에 따라 달라질 수 있으며, 현재로서는 정확한 예측을 제공하기 어렵습니다. 수출 전망은 경제 상황, 글로벌 시장 동향, 무역 정책, 산업별 성장률 등 다양한 요소에 영향을 받습니다. 

각국의 정부 기관이나 
[ RAG 답변 ] 2026년 수출 전망 금액은 7,952억 달러입니다.

[ Q ] 2030년 경제성장률 전망은?
[ 일반 LLM ] 2030년의 경제성장률 전망은 여러 요인에 따라 달라질 수 있습니다. 각국의 정부, 국제기구(예: IMF, 세계은행), 그리고 경제 연구 기관들이 다양한 경제 지표와 트렌드를 분석하여 예측을 제공합니다. 

일반적으로 경제성장률은 다음과 같은 요소에 영향을 받습니다:

[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
[ 일반 LLM ] 미국 연방준비제도(Federal Reserve, Fed)의 금리 결정 일정은 일반적으로

In [34]:
for q in questions:
    print(f"[ Q ] {q}")

    # 일반 LLM (RAG 없음)
    base_answer = parser.invoke(llm_base.invoke(q))
    print(f"[ 일반 LLM ] {base_answer[:150]}")

    # RAG 기반 LLM
    rag_answer = rag_chain.invoke(q)
    print(f"[ RAG 답변 ] {rag_answer[:150]}")
    print()

[ Q ] 주요 대외 불확실성 지표에 대해 설명해주세요.
[ 일반 LLM ] 대외 불확실성 지표는 국가나 지역의 경제에 영향을 미치는 외부 요인들의 불확실성을 측정하는 지표입니다. 이러한 지표는 경제 정책, 국제 무역, 정치적 사건, 자연재해 등 다양한 요소에 의해 영향을 받을 수 있습니다. 주요 대외 불확실성 지표에는 다음과 같은 것들이 있습
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 원/달러 환율 및 시장금리에 대해 설명해주세요.
[ 일반 LLM ] 원/달러 환율은 한국 원화(KRW)와 미국 달러(USD) 간의 교환 비율을 나타냅니다. 즉, 1달러를 얻기 위해 필요한 원화의 양을 의미합니다. 이 환율은 여러 요인에 의해 영향을 받으며, 경제 상황, 금리 차이, 무역 수지, 외환 시장의 수급 상황 등이 주요 요인으로
[ RAG 답변 ] 원/달러 환율은 거주자 해외증권투자와 엔화 동조 등으로 변동성이 높아진 가운데 최근 1,400원대 중반 수준으로 낮아졌습니다. 시장금리는 기준금리 인하 기대 약화와 수급 요인 등으로 상승세를 나타내고 있습니다.

[ Q ] AI과잉 투자에 대한 상황과 우려에 대해 자세하게 설명해주세요.
[ 일반 LLM ] AI(인공지능) 기술에 대한 과잉 투자는 최근 몇 년간 급격히 증가하고 있으며, 이는 여러 산업에서 AI의 잠재력을 활용하려는 기업과 투자자들의 관심을 반영합니다. 그러나 이러한 과잉 투자에는 몇 가지 우려가 따릅니다.

### 1. **기술의 성숙도 부족**
AI 기
[ RAG 답변 ] 보고서에서 확인되지 않습니다.

[ Q ] 2026년 2월 경제전망 요약표를 설명해주세요.
[ 일반 LLM ] 2026년 2월 경제전망 요약표는 특정 시점에서의 경제 지표와 예측을 요약한 자료로, 일반적으로 다음과 같은 항목들을 포함합니다:

1. **GDP 성장률**: 국가의 경제 성장 속도를 나타내며, 전년 대비 성장률을 예측합니다. 이는 경제의 전반적인 건강 상태를 보여줍
[ RAG 답변 ] 2026년 2월 경제전망

# 청크 갯수에 따른 성능 비교

In [35]:
NEW_INDEX = "finance-bok-test"
NS_A = "bok-nsa"  # chunk_size=200
NS_B = "bok-nsb"  # chunk_size=500
NS_C = "bok-nsc"  # chunk_size=1000

In [36]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=os.environ['PINECONE_API_KEY'])

if NEW_INDEX not in pc.list_indexes().names():
    pc.create_index(
        name=NEW_INDEX,
        dimension=1536,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print(f"{NEW_INDEX} 생성 완료")
else:
    print(f"{NEW_INDEX} 이미 존재함")

cmp_index = pc.Index(NEW_INDEX)
print(cmp_index.describe_index_stats())

finance-bok-test 생성 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [37]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "./data/2026년 2월 경제전망보고서(Indigo Book)_FFF.pdf"
loader = PyPDFLoader(PDF_PATH)
docs = loader.load()
print(f"총 페이지 수: {len(docs)}")

총 페이지 수: 93


In [38]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_a = RecursiveCharacterTextSplitter(chunk_size=200,  chunk_overlap=20)
splitter_b = RecursiveCharacterTextSplitter(chunk_size=500,  chunk_overlap=50)
splitter_c = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

splits_map = {
    NS_A: splitter_a.split_documents(docs),
    NS_B: splitter_b.split_documents(docs),
    NS_C: splitter_c.split_documents(docs),
}

for ns, splits in splits_map.items():
    avg_len = sum(len(s.page_content) for s in splits) // len(splits)
    print(f"{ns}: 청크 수 {len(splits)}개, 평균 길이 {avg_len}자")

bok-nsa: 청크 수 581개, 평균 길이 163자
bok-nsb: 청크 수 249개, 평균 길이 400자
bok-nsc: 청크 수 145개, 평균 길이 688자


In [39]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

for ns, splits in splits_map.items():
    print(f"{ns} 업서트 시작... ({len(splits)}개)")
    PineconeVectorStore.from_documents(
        documents=splits,
        embedding=embedding_model,
        index_name=NEW_INDEX,
        namespace=ns
    )
    print(f"{ns} 업서트 완료")

print("\n전체 완료")
print(cmp_index.describe_index_stats())


bok-nsa 업서트 시작... (581개)
bok-nsa 업서트 완료
bok-nsb 업서트 시작... (249개)
bok-nsb 업서트 완료
bok-nsc 업서트 시작... (145개)
bok-nsc 업서트 완료

전체 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'bok-nsa': {'vector_count': 581},
                'bok-nsb': {'vector_count': 249},
                'bok-nsc': {'vector_count': 145}},
 'total_vector_count': 975,
 'vector_type': 'dense'}


In [41]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

prompt = ChatPromptTemplate.from_template("""
당신은 한국은행 경제전망 보고서를 기반으로 답변하는 금융 전문 어시스턴트입니다.
아래 참고 문서를 바탕으로 질문에 정확하게 답하세요.
문서에 없는 내용은 "보고서에서 확인되지 않습니다"라고 답하세요.

[참고문서]
{context}

[질문]
{question}

한글로 간결하고 정확하게 답변하세요.
""")

def make_rag_chain(namespace):
    vectorstore = PineconeVectorStore.from_existing_index(
        index_name=NEW_INDEX,
        embedding=embedding_model,
        namespace=namespace
    )
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
    return (
        RunnableParallel(context=retriever, question=RunnablePassthrough())
        | prompt | llm | parser
    )

chain_a = make_rag_chain(NS_A)
chain_b = make_rag_chain(NS_B)
chain_c = make_rag_chain(NS_C)

print("RAG chain 생성 완료")

RAG chain 생성 완료


In [ ]:
test_questions = [
    # 수치 확인용
    "2026년 GDP 성장률 전망치는 얼마인가요?",
    "2026년 소비자물가 상승률 전망치는?",
    "2026년 수출 전망 금액은 얼마인가요?",

    # 범위 밖 질문 (환각 테스트)
    "2030년 경제성장률 전망은?",
    "미국 연방준비제도의 금리 결정 일정은?",

    # 맥락 이해 확인
    "성장률이 2%대로 반등하는 주요 원인은?",
    "부문별 온도차가 발생하는 이유는?"
]

In [42]:
for q in test_questions:
    print(f"[ Q ] {q}")
    print(f"  [A-200] {chain_a.invoke(q)[:150]}")
    print(f"  [B-500] {chain_b.invoke(q)[:150]}")
    print(f"  [C-1000] {chain_c.invoke(q)[:150]}")
    print()

[ Q ] 2026년 GDP 성장률 전망치는 얼마인가요?
  [A-200] 2026년 GDP 성장률 전망치는 2.0%입니다.
  [B-500] 2026년 GDP 성장률 전망치는 2.0%입니다.
  [C-1000] 2026년 GDP 성장률 전망치는 2.0%입니다.

[ Q ] 2026년 소비자물가 상승률 전망치는?
  [A-200] 2026년 소비자물가 상승률 전망치는 2.2%입니다.
  [B-500] 2026년 소비자물가 상승률 전망치는 2.0%입니다.
  [C-1000] 2026년 소비자물가 상승률 전망치는 2.0%입니다.

[ Q ] 2026년 수출 전망 금액은 얼마인가요?
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 2026년 수출 전망 금액은 7,952억 달러입니다.
  [C-1000] 2026년 수출 전망 금액은 7,952억 달러입니다.

[ Q ] 2030년 경제성장률 전망은?
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 보고서에서 확인되지 않습니다.
  [C-1000] 보고서에서 확인되지 않습니다.

[ Q ] 미국 연방준비제도의 금리 결정 일정은?
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 보고서에서 확인되지 않습니다.
  [C-1000] 보고서에서 확인되지 않습니다.

[ Q ] 성장률이 2%대로 반등하는 주요 원인은?
  [A-200] 성장률이 2%대로 반등하는 주요 원인은 보고서에서 확인되지 않습니다.
  [B-500] 보고서에서 확인되지 않습니다.
  [C-1000] 보고서에서 확인되지 않습니다.

[ Q ] 부문별 온도차가 발생하는 이유는?
  [A-200] 부문별 온도차가 발생하는 이유는 부문별 성장 차별화가 심화되기 때문입니다. 같은 정도의 성장을 하더라도 부문별로 성장 차별이 심화될수록 소비경로와 물가에 미치는 영향이 달라지기 때문입니다.
  [B-500] 부문별 온도차는 주로 IT 대기업이 주도하는 K자형 회복국면에서 발생하며, 이는 일부 부문(예: 반도체)에서의 

In [43]:
for q in questions:
    print(f"[ Q ] {q}")
    print(f"  [A-200] {chain_a.invoke(q)[:150]}")
    print(f"  [B-500] {chain_b.invoke(q)[:150]}")
    print(f"  [C-1000] {chain_c.invoke(q)[:150]}")
    print()

[ Q ] 주요 대외 불확실성 지표에 대해 설명해주세요.
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 보고서에서 확인되지 않습니다.
  [C-1000] 보고서에서 확인되지 않습니다.

[ Q ] 원/달러 환율 및 시장금리에 대해 설명해주세요.
  [A-200] 원/달러 환율은 최근 1,400원 중반에서 높은 변동성을 보이고 있으며, 거주자 해외증권투자와 엔화 동조 등의 요인으로 변동성이 증가했습니다. 시장금리는 기준금리 인하 기대 약화와 수급 요인 등으로 상승세를 나타내고 있습니다.
  [B-500] 원/달러 환율은 거주자 해외증권투자와 엔화 동조 등으로 변동성이 높아진 가운데 최근 1,400원대 중반 수준으로 낮아졌습니다. 시장금리는 기준금리 인하 기대 약화와 수급 요인 등으로 상승세를 나타내고 있습니다.
  [C-1000] 원/달러 환율은 거주자 해외투자 및 외국인 국내주식 매도 등 상방 요인과 미 달러화 약세 등 하방 요인이 혼재하여 1,400원 중반에서 높은 변동성을 보이고 있습니다. 

시장금리는 국내외 통화정책 기대 변화 및 주요국 재정 확대 우려로 큰 폭 상승하였으며, 국고채 금

[ Q ] AI과잉 투자에 대한 상황과 우려에 대해 자세하게 설명해주세요.
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 보고서에서 확인되지 않습니다.
  [C-1000] 보고서에서 확인되지 않습니다.

[ Q ] 2026년 2월 경제전망 요약표를 설명해주세요.
  [A-200] 보고서에서 확인되지 않습니다.
  [B-500] 2026년 2월 경제전망 요약표는 다음과 같습니다:

- **GDP 성장률**: 2024년 2.0%, 2025년 1.0%, 2026년 2.0% (+0.2), 2027년 1.8% (-0.1)
- **소비자물가 상승률**: 2024년 2.3%, 2025년 2.1%, 202
  [C-1000] 2026년 2월 경제전망 요약표는 다음과 같은 주요 내용을 포함하고 있습니다:

1. **세계경제 성장률**: 2024년